# Débat multi-agents vs vote majoritaire — expériences sur GPU (Kaggle)

Ce notebook lance les étapes d'expériences de [multi-agent-debate-math](https://github.com/Touayouch/AI-projects/tree/main/multi-agent-debate-math)
avec **vLLM** sur les GPU gratuits de Kaggle.

**Réglages Kaggle (panneau de droite) :**
1. *Accelerator* : **GPU T4 x2** (le P100 n'est pas pris en charge par vLLM).
2. *Internet* : **On** (nécessite un compte dont le numéro de téléphone est vérifié).
3. **Dépôt privé uniquement** : *Add-ons → Secrets*, ajouter un secret `GITHUB_TOKEN`
   contenant un jeton GitHub en lecture seule sur ce dépôt (*Settings → Developer settings →
   Fine-grained tokens*, permission *Contents : Read-only*). Inutile si le dépôt est public.
4. Lancer avec **Save Version → Save & Run All (Commit)** : le notebook tourne en arrière-plan
   jusqu'à 12 h, même navigateur fermé, et ses sorties sont conservées.

**Reprise après une session interrompue :** *Add Input → Your Work → Notebooks*, ajouter la
version précédente de ce notebook. Ses résultats sont copiés au démarrage et seuls les
problèmes manquants sont traités.

**Important :** toujours partir d'une session neuve (*Run → Restart & clear* ne suffit pas, il
faut arrêter la session) : les paquets installés par une session précédente restent en place.

**Ordre conseillé :** session 1 : `STAGES = ["gsm8k", "math500"]` (un GPU chacun) ;
session 2 : `STAGES = ["hetero"]` (utilise les deux GPU).

In [ ]:
STAGES = ["gsm8k", "math500"]  # étapes de configs/stages.yaml, lancées en parallèle
REPO = "https://github.com/Touayouch/AI-projects.git"
BRANCH = "main"
MAX_CONCURRENCY = 128  # requêtes simultanées par étape (vLLM les regroupe en batch)

In [ ]:
!nvidia-smi  # la ligne « CUDA Version » indique la version maximale gérée par le pilote
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

## Installation

In [ ]:
# La version PyPI de vLLM 0.32 est compilée pour CUDA 13 (libcudart.so.13), trop récente
# pour les pilotes de Kaggle. On installe la version officielle compilée pour CUDA 12.9,
# avec PyTorch de l'index cu129 (versions figées pour la reproductibilité).
VLLM_VERSION = "0.32.0"
VLLM_WHEEL = (f"https://github.com/vllm-project/vllm/releases/download/v{VLLM_VERSION}/"
              f"vllm-{VLLM_VERSION}+cu129-cp38-abi3-manylinux_2_28_x86_64.whl")
!pip install -q "{VLLM_WHEEL}" --extra-index-url https://download.pytorch.org/whl/cu129

In [ ]:
import subprocess
print(subprocess.run(["python", "-c", "import torch; print('torch', torch.__version__, '| CUDA', "
                      "torch.version.cuda, '| GPU dispo :', torch.cuda.is_available(), '| capacité', "
                      "torch.cuda.get_device_capability() if torch.cuda.is_available() else None)"],
                     capture_output=True, text=True).stdout)

In [ ]:
import os, subprocess
os.chdir("/kaggle/working")
def repo_url() -> str:
    """URL de clonage ; avec le secret Kaggle GITHUB_TOKEN si le dépôt est privé."""
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    except Exception:
        return REPO
    return REPO.replace("https://", f"https://x-access-token:{token}@")

if not os.path.exists("AI-projects"):
    # capture_output : en cas d'échec, ne pas afficher l'URL contenant le jeton
    result = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, repo_url()],
                            capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError("Échec du clonage : dépôt privé sans secret GITHUB_TOKEN valide, "
                           "ou branche introuvable")
    # le jeton ne doit pas rester dans la configuration du clone
    subprocess.run(["git", "-C", "AI-projects", "remote", "set-url", "origin", REPO], check=True)
os.chdir("/kaggle/working/AI-projects/multi-agent-debate-math")
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)
!vllm --version
!debate check-grader

## Restauration des résultats

Les résultats vivent dans `/kaggle/working/results` (sortie du notebook), relié au dossier
`results/` du projet. On y copie les résultats déjà versionnés dans le dépôt et ceux d'une
version précédente du notebook ajoutée en entrée, puis on décompresse les `.jsonl.gz`.

In [ ]:
import gzip, shutil
from pathlib import Path

out = Path("/kaggle/working/results")
out.mkdir(exist_ok=True)
project_results = Path("results")
sources = [p for p in Path("/kaggle/input").glob("**/results") if p.is_dir()]
if project_results.is_dir() and not project_results.is_symlink():
    sources.append(project_results)
for source in sources:
    for f in source.iterdir():
        if f.is_file() and not (out / f.name).exists():
            shutil.copy(f, out / f.name)
if project_results.is_dir() and not project_results.is_symlink():
    shutil.rmtree(project_results)
if not project_results.exists():
    project_results.symlink_to(out)

for gz in out.glob("*.jsonl.gz"):
    plain = gz.with_suffix("")
    if not plain.exists():
        with gzip.open(gz, "rb") as src, plain.open("wb") as dst:
            shutil.copyfileobj(src, dst)
for f in sorted(out.glob("*.jsonl")):
    print(f.name, sum(1 for _ in f.open()), "problèmes")

## Lancement des étapes (une par GPU, en parallèle)

In [ ]:
import time

env = os.environ | {"MAX_CONCURRENCY": str(MAX_CONCURRENCY), "LOG_LEVEL": "INFO"}
procs = {}
for stage in STAGES:
    log = open(f"/kaggle/working/stage_{stage}.log", "a")
    procs[stage] = subprocess.Popen(["debate", "stage", stage], stdout=log,
                                    stderr=subprocess.STDOUT, env=env)

start = time.time()
while any(p.poll() is None for p in procs.values()):
    time.sleep(600)
    print(f"--- {(time.time() - start) / 3600:.1f} h")
    for stage in STAGES:
        lines = open(f"/kaggle/working/stage_{stage}.log").read().splitlines()
        progress = [l for l in lines if "debate.experiment" in l]
        print(f"[{stage}]", progress[-1] if progress else (lines[-1] if lines else ""))

for stage, p in procs.items():
    print(stage, "terminée" if p.returncode == 0 else f"ÉCHEC (code {p.returncode})")

## Résumés et fichiers à récupérer

Les fichiers `results_gz/*.jsonl.gz` sont à télécharger (onglet *Output*) puis à placer dans
`multi-agent-debate-math/results/` du dépôt avant de les commiter.

In [ ]:
gz_dir = Path("/kaggle/working/results_gz")
gz_dir.mkdir(exist_ok=True)
for f in sorted(out.glob("*.jsonl")):
    with f.open("rb") as src, gzip.open(gz_dir / (f.name + ".gz"), "wb") as dst:
        shutil.copyfileobj(src, dst)
    shutil.copy(f.with_suffix(".config.yaml"), gz_dir)
    !debate report {f}